# AgentIR: candidate memory before question generation

**New notebook, separate results. Reuses the previous data bundle in Drive.** Use a Colab T4 GPU. No Qwen, API key or new dialogue is needed. BLIP only embeds saved descriptions; image embeddings are already cached.

**Question:** Do earlier search results provide useful candidates that the current search misses, and can a small evidence set retain them?

We reconstruct earlier rankings from saved PlugIR summaries. We do not have the original search-result logs or original Q&A. The original description is available, but later summaries are not a verified record of user facts.

At stages **after 4 and 9 answers**, every method receives the same **50-image pool budget** (primary analysis), then **100 images** (sensitivity check). Each pool also produces exactly **10 evidence images** through the same diagnostic selector.

| Method | Candidate pool |
|---|---|
| Current | Top results for the current summary |
| Original only | Top results for the initial description; missing control from our first pilot |
| Original + current RRF | Equal-weight fusion of those two rankings |
| Previous + current RRF | Fusion of the immediately preceding and current summaries |
| All history RRF | Fusion of every unique description from the initial query through the current stage |
| History quota | Half current top results, then unique historical candidates taken round-robin, newest history first |

RRF uses k=60. History quota consults each earlier top-B list, fills duplicates from other lists and then current results if necessary. It never enlarges the final budget.

**Selection diagnostic:** MMR balances current-query relevance and image-vector diversity with a fixed relevance weight of 0.7. All methods use the same rule. This is **not** PlugIR's clustering or A2R's DPP. We also record each pool's first 10 images as an ordering control.

Target IDs enter evaluation only, never candidate construction or selection. Higher target coverage does not establish better questions. We cannot automatically measure contradictions with user-confirmed facts without the Q&A.

**Files:** this notebook + `candidate_memory_tools.zip`. Your previous `multi_query_pilot_bundle.zip` should already be in Drive. Teammates without it need that ZIP too.


## 1. Install packages
Select Runtime → Change runtime type → T4 GPU, then run cells in order. If Colab requests a restart, restart and rerun this cell.

In [ ]:
%pip -q install "transformers==4.51.3" "huggingface-hub==0.36.2" "pandas>=2,<4" matplotlib pillow requests


## 2. Connect Drive
Use your own Drive results folder. Reuse RUN_NAME to resume, but choose a new name if changing the experiment code or settings.

In [ ]:
import os
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['HF_HOME']='/content/agentir_model_cache'
from pathlib import Path
from google.colab import drive, files
drive.mount('/content/drive')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing.')
print('GPU:',torch.cuda.get_device_name(0))
PROJECT_DIR=Path('/content/drive/MyDrive/AgentIR_candidate_memory')
RUN_NAME='memory_v1'
RUN_DIR=PROJECT_DIR/'runs'/RUN_NAME
RUN_DIR.mkdir(parents=True,exist_ok=True)
DATA_DIR=Path('/content/agentir_memory_data')
WORK_DIR=Path('/content/agentir_memory_tools')
DATA_DIR.mkdir(exist_ok=True)
WORK_DIR.mkdir(exist_ok=True)


## 3. Load tools and reuse the previous data
Upload `candidate_memory_tools.zip` when prompted. The old data ZIP is reused automatically from the previous notebook’s Drive folder. If it is missing, upload it when asked.

In [ ]:
import zipfile,hashlib,json

def load_zip(path,expected,destination):
    if not path.exists():
        print('Please upload:',path.name)
        uploaded=files.upload()
        matches=[v for n,v in uploaded.items() if n==path.name]
        if len(matches)!=1:raise ValueError('Expected '+path.name)
        if hashlib.sha256(matches[0]).hexdigest()!=expected:raise ValueError('ZIP version mismatch')
        path.parent.mkdir(parents=True,exist_ok=True)
        path.write_bytes(matches[0])
    if hashlib.sha256(path.read_bytes()).hexdigest()!=expected:raise ValueError('ZIP version mismatch: '+str(path))
    with zipfile.ZipFile(path) as z:
        if any(Path(n).name!=n for n in z.namelist()):raise ValueError('Unexpected ZIP paths')
        z.extractall(destination)

load_zip(Path('/content/drive/MyDrive/AgentIR_multi_query/multi_query_pilot_bundle.zip'),'4418ddc07374ab4b42d1c0b97f034afc3580e9adf28d76e6f81da2087554b9db',DATA_DIR)
load_zip(PROJECT_DIR/'candidate_memory_tools.zip','ca83a71d6f29c1561c0a4c2019b72eddf96a6e736392dca8748a6ed68be652e5',WORK_DIR)
manifest=json.loads((WORK_DIR/'memory_manifest.json').read_text())
print({s:len(ids) for s,ids in manifest['splits'].items()})


## 4. Run the 10-target setup check
This reuses the earlier smoke targets. It checks data loading, candidate construction and output formatting. It does not generate new text. First use downloads the BLIP model if it is not already cached. Progress appears below and the full log is saved to Drive.

In [ ]:
import subprocess,sys,time

def run_phase(split):
    folder=RUN_DIR/split
    folder.mkdir(exist_ok=True)
    command=[sys.executable,'-u',str(WORK_DIR/'memory_experiment.py'),
             '--data',str(DATA_DIR),'--output',str(folder),'--split',split]
    started=time.perf_counter()
    with (folder/'run.log').open('a') as log:
        proc=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in proc.stdout:
            print(line,end='');log.write(line);log.flush()
        status=proc.wait()
    if status:raise RuntimeError(f'Run stopped. See the actual error above or {folder / "run.log"}. Cached text embeddings are preserved.')
    print(f'Completed in {(time.perf_counter()-started)/60:.1f} minutes')
    return folder

SMOKE_DIR=run_phase('smoke')
import pandas as pd
display(pd.read_csv(SMOKE_DIR/'summary.csv').query('budget == 50'))


## 5. Run the 100-target development comparison
These are the same 100 targets from the previous pilot. They are now development data, not fresh validation. Expect a shorter run than Qwen generation, but model download and Colab hardware affect timing. Text embeddings are checkpointed, so rerunning after interruption reuses completed encoding.

In [ ]:
DEV_DIR=run_phase('development')


## 6. Read the numbers

**Start with `control_summary`:** does original-only beat the current summary or their fusion?

**Pool coverage:** percentage of targets included in the final 50 or 100 candidates. **Selected-10 coverage:** percentage included in the 10 images chosen by MMR. Higher is better for target coverage, but does not prove that the other images support informative questions.

**Recovered / lost:** targets newly included / excluded versus current-only at the same stage and budget. Intervals are paired bootstrap estimates, not adjusted for the many exploratory comparisons.

If history improves pool coverage but not selected-10 coverage, selection may be losing the recovered evidence. If it hurts both, retaining old candidates may waste capacity. Compare all-history against original + current to see whether intermediate rounds add value. Never compare a 100-image method against a 50-image baseline as evidence for memory.


In [ ]:
display(pd.read_csv(DEV_DIR/'control_summary.csv').round(3))
summary=pd.read_csv(DEV_DIR/'summary.csv')
comparisons=pd.read_csv(DEV_DIR/'comparisons.csv')
display(summary.query('budget == 50').round(2))
display(comparisons.query('budget == 50 and baseline == "current"').round(2))
print('Does intermediate history add value beyond original + current?')
display(comparisons.query('budget == 50 and baseline == "original_current_rrf" and method in ["all_history_rrf", "history_quota", "previous_current_rrf"]').round(2))


In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(15,5),sharey=True)
for ax,t in zip(axes,[4,9]):
    sub=summary.query('budget == 50 and answers == @t').set_index('method')
    sub[['pool_coverage_pct','selected10_coverage_pct']].plot.bar(ax=ax,color=['#126F6B','#5D70A6'])
    ax.set_title(f'After {t} answers: 50-image pool, 10 selected')
    ax.set_ylabel('Targets covered (%)');ax.set_ylim(0,100)
    ax.set_xlabel('');ax.legend(['Pool coverage','Selected-10 coverage'])
plt.tight_layout()
fig.savefig(DEV_DIR/'coverage.png',dpi=160,bbox_inches='tight')
plt.show()


## 7. Inspect gains, losses and the selected images

The table lists cases where pool membership differs from current-only at budget 50. It deliberately includes both gains and losses. Set SOURCE_INDEX, ANSWERS and METHOD to inspect a case.

The optional image cell fetches the target and selected COCO thumbnails from public COCO image URLs. These are for human inspection only and never enter selection. Failed image downloads do not affect numerical results. Ask: are older candidates still plausible, do they expose different unknown attributes, and would a question based on them help? Without original Q&A, do not label a detail as user-confirmed.


In [ ]:
detail=pd.read_csv(DEV_DIR/'per_target.csv')
base=detail.query('method == "current"')[['source_index','answers','budget','target_in_pool','target_in_selected10']]
paired=detail.merge(base,on=['source_index','answers','budget'],suffixes=('','_current'))
paired['pool_change']=paired.target_in_pool-paired.target_in_pool_current
paired['selected_change']=paired.target_in_selected10-paired.target_in_selected10_current
cases=paired.query('budget == 50 and method != "current" and (pool_change != 0 or selected_change != 0)')
display(cases[['source_index','answers','method','pool_change','selected_change']])
examples=json.loads((DEV_DIR/'examples.json').read_text())
SOURCE_INDEX=manifest['splits']['development'][0]  # replace using a row above
ANSWERS=9
METHOD='history_quota'
example=next(e for e in examples if e['source_index']==SOURCE_INDEX and e['answers']==ANSWERS and e['budget']==50 and e['method']==METHOD)
print('Original:',example['original'])
print('Current:',example['current'])
display(pd.DataFrame(example['history']))
print('Target in pool:',bool(example['target_in_pool']),'| in selected 10:',bool(example['target_in_selected10']))


In [ ]:
# Optional: download and display image evidence, without calling any agent.
import requests,io
from PIL import Image
fig,axes=plt.subplots(3,5,figsize=(16,10))
for ax in axes.flat:ax.axis('off')
paths=[example['target_path']]+example['selected_paths']
for j,(path,ax) in enumerate(zip(paths,axes.flat)):
    try:
        url='https://images.cocodataset.org/'+path
        response=requests.get(url,timeout=20);response.raise_for_status()
        ax.imshow(Image.open(io.BytesIO(response.content)))
    except Exception as error:
        ax.text(.05,.5,'Image unavailable',transform=ax.transAxes)
        print(path, str(error)[:160])
    if j==0:label='Target (evaluation only)'
    else:
        target_mark=' [TARGET]' if example['selected_ids'][j-1]==example['target_id'] else ''
        label=f"Selected {j}: current rank {example['selected_current_ranks'][j-1]}{target_mark}"
    ax.set_title(label,fontsize=10)
plt.tight_layout();plt.show()
fig.savefig(DEV_DIR/f'inspection_{SOURCE_INDEX}_{ANSWERS}_{METHOD}.png',dpi=140,bbox_inches='tight')


## 8. Reserved holdout (leave off during exploration)
The manifest reserves 100 fresh targets with no image overlap with development, smoke or earlier inspected samples. Freeze the settings and primary comparison before running this. If you use these results to change the method, this set becomes development data too. The primary budget is 50, primary stage is after 9 answers, and the main memory comparison is all-history versus original + current; current-only remains the reference. Both pool and selected-10 coverage matter. This is a proposal for validation, not a claim of preregistration.


In [ ]:
RUN_HOLDOUT=False
if RUN_HOLDOUT:
    HOLDOUT_DIR=run_phase('holdout')
    display(pd.read_csv(HOLDOUT_DIR/'summary.csv').query('budget == 50'))
else:
    print('Holdout left untouched. Inspect development results first.')


## 9. Download results
Send this ZIP back. It includes numerical summaries, per-target membership, query history, exact selected-image IDs, settings, logs and any inspection figures you generated. It does not include the large image cache or downloaded model weights.

In [ ]:
import shutil
archive=shutil.make_archive('/content/AgentIR_candidate_memory_results','zip',RUN_DIR)
files.download(archive)
